#### Measurement Framework Library - MFPortal
# Create a User Experiment 
This notebook creates a small 3 node experiment with an added measurement network that will be used to connect a measurement node to the slice.

**Before running this notebook**, confirm `mflib` is at version 1.0.10. Install it with `pip install fabrictestbed-mflib==1.0.10`, or run [InstallMFlib-PortalVersion.ipynb](InstallMFlib-PortalVersion.ipynb), included alongside this one. STEP 1 below prints the installed version so you can double-check it before adding the measurement network.

See also [Chatbox-MCP-Quick-Start.ipynb](Chatbox-MCP-Quick-Start.ipynb) (also in this directory) for connecting an AI assistant to the MCP endpoint this notebook's slice registration produces.

In [ ]:
slice_name = "MeasNetSlice"

In [ ]:
# Note when this notebook was actually run.
from datetime import datetime
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

------------------
# STEP 1: General Imports
------------------

In [ ]:
%%time
import os
import json
import traceback

### Import mflib and check version

In [ ]:
import mflib
print(f"MFLib version  {mflib.__version__} ")
from mflib.mfportal import MFPortal

------------------
# STEP 2: Setup Experiment Slice
------------------

### Import fablib

In [ ]:
%%time
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager

fablib = fablib_manager()

#fablib.show_config();


### Set Basic Slice Information

In [ ]:
%%time

# Get 3 random sites
[site1,site2,site3] = fablib.get_random_sites(count=3)

node1_name = 'Node1'
node2_name = 'Node2'
node3_name = 'Node3'

network1_name='net1'
network2_name='net2'
network3_name='net3'

node1_nic_name = 'nic1'
node2_nic_name = 'nic2'
node3_nic_name = 'nic3'

print(f"Setting up slice {slice_name}")
print(f"Using sites {site1}, {site2}, {site3}")

### Create Experiment Topology

In [ ]:
%%time
try:
    # Create Slice
    slice = fablib.new_slice(name=slice_name)
    
    # Node1
    node1 = slice.add_node(name=node1_name, site=site1)
    iface1 = node1.add_component(model='NIC_Basic', name=node1_nic_name).get_interfaces()[0]
    
    # Node2
    node2 = slice.add_node(name=node2_name, site=site2)
    iface2  = node2.add_component(model='NIC_Basic', name=node2_nic_name).get_interfaces()[0]

    # Node3
    node3 = slice.add_node(name=node3_name, site=site3)
    iface3  = node3.add_component(model='NIC_Basic', name=node3_nic_name).get_interfaces()[0]

    # Networks
    net1 = slice.add_l3network(name=network1_name, interfaces=[iface1], type='IPv4')
    net2 = slice.add_l3network(name=network2_name, interfaces=[iface2], type='IPv4')
    net3 = slice.add_l3network(name=network3_name, interfaces=[iface3], type='IPv4')
    
    print(f"Slice Topology Done.")
except Exception as e:
    print(f"Exception: {e}")

![Slice topology: three nodes each connected to an Experiment Network](images/slice-topology.png)

------------------
# STEP 3: Add a Measurement Network using MFLib
------------------
This must be done **before** the slice is submitted. `MFPortal.add_meas_network_presubmit()` is the supported way to do this today -- it wires a FABNetv6 measurement-network NIC onto every node while the slice is still just a local topology.

In [ ]:
wired = MFPortal.add_meas_network_presubmit(slice)
print(f"Wired FABNetv6 NIC(s) for: {wired}")

------------------
# STEP 4: Submit the Slice
------------------

In [ ]:
%%time
try:
    # Submit Slice Request
    print(f'Submitting the new slice, "{slice_name}"...')
    slice.submit()
    print(f'{slice_name} creation done.')

except Exception as e:
    print(f"Slice Fail: {e}")
    traceback.print_exc()

In [ ]:
%%time
print("Slice creation completed")

------------------
# STEP 5: Register Slice with MFPortal
------------------
Once the slice has been created and has a measurement network, register the slice with the MFPortal.

In [ ]:
response = MFPortal.register_slice_with_portal(slice)

The URL to access the slice on the MFPortal is included in the register response. Click on that link and you will be taken to the slice's measurement status page. Included on the page is the measurement node's status, the Grafana URL, and the MCP configuration.

To connect an AI assistant (e.g. Chatbox) to the MCP endpoint shown on that page, see [Chatbox-MCP-Quick-Start.ipynb](Chatbox-MCP-Quick-Start.ipynb) in this directory.

![Slice status page on the MFPortal, showing Meas Node health, Route-Watcher SSH status, Grafana login, and the MCP endpoint/config](images/slice-status-page.png)

In [ ]:
print('--------------------------------------------------------------------------------------------------------------------------------')
print(f'Open the following link to access the status, view MCP configuration and to see the Grafana link.  {response["view_url"]}')
print('--------------------------------------------------------------------------------------------------------------------------------')

In [ ]:
# Note when this notebook finishes the run.
from datetime import datetime
print(datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

## Further reading

- [InstallMFlib-PortalVersion.ipynb](InstallMFlib-PortalVersion.ipynb) -- installs the specific `mflib` version this notebook requires.
- [Chatbox-MCP-Quick-Start.ipynb](Chatbox-MCP-Quick-Start.ipynb) -- connect an AI assistant (Chatbox) to the MCP endpoint from Step 5's registration response.